# Step 3 — Verify Search Reliability with O'Byrne (2015); Produce Final Dataset

## Purpose
Step 2 produced adoption years from web searches. **This step asks: how reliable is that search data?**

O'Byrne (2015) provides a fully independent binary classification — adopted/not adopted by 2012 —
derived from a systematic academic survey of US cities. We use it as an external audit layer.

**Sub-goal 3A — Reliability audit:** Confusion matrix, scalar metrics, conflict listing, source-type
breakdown, year plausibility check.

**Sub-goal 3B — Final dataset:** Merge verified search results with O'Byrne labels.

## Inputs
- `data_clean/search_results.csv` (314 rows, from step2_parse_search_log.ipynb)
- `data_clean/obyrne_extracted.csv` (242 rows, from step2_extract_obyrne.ipynb)
- `data_clean/city_list.csv` (314 rows, for population and GEOID reference)

## Outputs
- `data_clean/search_audit_metrics.csv` — scalar audit metrics (raw + corrected)
- `data_clean/search_audit.csv` — per-city audit flags (314 rows)
- `data_clean/city_311_final.csv` — 314-row analysis-ready dataset

In [1]:
import re
import pandas as pd
from difflib import get_close_matches

CLEAN = '../data_clean/'

search   = pd.read_csv(CLEAN + 'search_results.csv',  dtype={'GEOID': str})
obyrne   = pd.read_csv(CLEAN + 'obyrne_extracted.csv')
city_list = pd.read_csv(CLEAN + 'city_list.csv',      dtype={'GEOID': str})

print(f'search_results  : {len(search)} rows')
print(f'obyrne_extracted: {len(obyrne)} rows')
print(f'city_list       : {len(city_list)} rows')
print()
print('O\'Byrne adopt_by2012 distribution:')
print(obyrne['adopt_by2012'].value_counts(dropna=False).to_string())

search_results  : 314 rows
obyrne_extracted: 242 rows
city_list       : 314 rows

O'Byrne adopt_by2012 distribution:
adopt_by2012
0.0    180
1.0     53
NaN      9


## 3B-1. Match O'Byrne cities to city_list GEOID

In [2]:
SUFFIXES = re.compile(
    r'\b(city|town|village|borough|municipality|county|metro(politan)?'
    r'|government|consolidated|unified|urban|charter)\b',
    re.IGNORECASE
)

def norm(name):
    name = SUFFIXES.sub('', str(name))
    name = re.sub(r'[^a-z\s]', '', name.lower())
    return re.sub(r'\s+', ' ', name).strip()

city_list['name_norm'] = city_list['place_name'].apply(norm)

# Build lookup: (state_abbr, name_norm) -> GEOID
lookup = (
    city_list.groupby(['state_abbr', 'name_norm'])['GEOID']
    .first().reset_index()
    .set_index(['state_abbr', 'name_norm'])['GEOID'].to_dict()
)
state_names = city_list.groupby('state_abbr')['name_norm'].apply(list).to_dict()

# Parse O'Byrne city_raw: "City, ST" or "City, ST (note)"
def parse_obyrne_city(raw):
    raw = re.sub(r'\(.*?\)', '', str(raw)).strip()
    parts = raw.split(',')
    if len(parts) >= 2:
        city_part = parts[0].strip()
        state_part = parts[-1].strip().split()[0]  # take first word (state abbr)
        return city_part, state_part
    return raw, ''

obyrne[['ob_city', 'ob_state']] = obyrne['city_raw'].apply(
    lambda x: pd.Series(parse_obyrne_city(x))
)
obyrne['name_norm'] = obyrne['ob_city'].apply(norm)

def match_obyrne(row):
    key = (row['ob_state'], row['name_norm'])
    if key in lookup:
        return lookup[key], 'exact'
    cands = state_names.get(row['ob_state'], [])
    m = get_close_matches(row['name_norm'], cands, n=1, cutoff=0.82)
    if m:
        return lookup[(row['ob_state'], m[0])], f'fuzzy:{m[0]}'
    return None, 'no_match'

obyrne[['GEOID', 'ob_match_type']] = obyrne.apply(match_obyrne, axis=1, result_type='expand')

print('O\'Byrne match summary:')
print(obyrne['ob_match_type'].str.split(':').str[0].value_counts().to_string())

unmatched = obyrne[obyrne['GEOID'].isna()]
if len(unmatched):
    print(f'\nUnmatched O\'Byrne cities ({len(unmatched)}):')
    print(unmatched[['city_raw', 'ob_state', 'adopt_by2012']].to_string(index=False))

O'Byrne match summary:
ob_match_type
exact       226
no_match     16

Unmatched O'Byrne cities (16):
                                        city_raw ob_state  adopt_by2012
Indianapolis, IN (City portion of Marion County)       IN           0.0
                                  Washington, DC       DC           1.0
                     Metropolitan Government, TN       TN           1.0
            Miami-Dade County, FL (City portion)       FL           1.0
                                    Honolulu, HI       HI           0.0
                                  Louisville, KY       KY           1.0
                                 Baton Rouge, LA       LA           1.0
      Augusta-Richmond County, GA (City portion)       GA           0.0
                                  Government, KS       KS           1.0
                                       Flint, MI       MI           0.0
                                     Amherst, NY       NY           0.0
                                   

## 3B-2. Merge O'Byrne into search results

In [3]:
# Keep one O'Byrne row per GEOID (adopt_by2012=1 takes precedence if duplicated)
ob_dedup = (
    obyrne[obyrne['GEOID'].notna()]
    .sort_values('adopt_by2012', ascending=False)
    .drop_duplicates('GEOID', keep='first')
    [['GEOID', 'adopt_by2012', 'ob_match_type']]
)

merged = search.merge(ob_dedup, on='GEOID', how='left')

# Build obyrne_label
def obyrne_label(row):
    if pd.isna(row['adopt_by2012']):
        if pd.isna(row.get('ob_match_type')):
            return 'not_in_obyrne'
        return 'unclear'
    if row['adopt_by2012'] == 1:
        return 'adopted_by2012'
    if row['adopt_by2012'] == 0:
        return 'not_adopted_by2012'
    return 'unclear'

# Cities not matched to O'Byrne have NaN GEOID in ob_dedup → adopt_by2012 = NaN after left join
# Distinguish 'not_in_obyrne' (never matched) from 'unclear' (matched but adopt_by2012=NaN)
merged['in_obyrne'] = merged['GEOID'].isin(ob_dedup['GEOID'])

def obyrne_label2(row):
    if not row['in_obyrne']:
        return 'not_in_obyrne'
    if pd.isna(row['adopt_by2012']):
        return 'unclear'
    return 'adopted_by2012' if row['adopt_by2012'] == 1 else 'not_adopted_by2012'

merged['obyrne_label'] = merged.apply(obyrne_label2, axis=1)

print('obyrne_label distribution:')
print(merged['obyrne_label'].value_counts().to_string())

obyrne_label distribution:
obyrne_label
not_adopted_by2012    170
not_in_obyrne          88
adopted_by2012         47
unclear                 9


## 3A. Reliability Audit

Restrict the confusion matrix to cities in **both** datasets where O'Byrne coverage applies
(i.e., `obyrne_label` is `adopted_by2012` or `not_adopted_by2012`).

For the matrix, we classify search result as:
- **year_found**: adoption_year_raw is a 4-digit year integer string
- **no_311**: adoption_year_raw == 'no_311'
- **unknown**: adoption_year_raw == 'unknown'

In [4]:
def search_class(val):
    s = str(val).strip().lower()
    if s == 'no_311':
        return 'no_311'
    if s == 'unknown':
        return 'unknown'
    if re.match(r'^\d{4}$', s):
        return 'year_found'
    return 'unknown'

merged['search_class'] = merged['adoption_year_raw'].apply(search_class)

# Subset for confusion matrix: only cities with definitive O'Byrne label
audit_sub = merged[merged['obyrne_label'].isin(['adopted_by2012', 'not_adopted_by2012'])].copy()

print(f'Cities in confusion matrix subset: {len(audit_sub)}')
print()

# Confusion matrix
cm = pd.crosstab(
    audit_sub['obyrne_label'],
    audit_sub['search_class'],
    margins=True
)
print('Confusion matrix (rows=O\'Byrne, cols=search result):')
print(cm.to_string())

Cities in confusion matrix subset: 217

Confusion matrix (rows=O'Byrne, cols=search result):
search_class        no_311  unknown  year_found  All
obyrne_label                                        
adopted_by2012           0        5          42   47
not_adopted_by2012      39       75          56  170
All                     39       80          98  217


In [5]:
# TP/TN/FP/FN — restrict to year_found vs no_311 for binary metrics
# (unknown rows are ambiguous and excluded from binary scalar metrics)
binary = audit_sub[audit_sub['search_class'].isin(['year_found', 'no_311'])].copy()

TP = len(binary[(binary['obyrne_label'] == 'adopted_by2012')    & (binary['search_class'] == 'year_found')])
TN = len(binary[(binary['obyrne_label'] == 'not_adopted_by2012') & (binary['search_class'] == 'no_311')])
FP = len(binary[(binary['obyrne_label'] == 'not_adopted_by2012') & (binary['search_class'] == 'year_found')])
FN = len(binary[(binary['obyrne_label'] == 'adopted_by2012')    & (binary['search_class'] == 'no_311')])

total = TP + TN + FP + FN
agreement_rate = (TP + TN) / total if total > 0 else float('nan')
conflict_rate  = (FP + FN) / total if total > 0 else float('nan')
fn_rate        = FN / (TP + FN)   if (TP + FN) > 0 else float('nan')
fp_rate        = FP / (TN + FP)   if (TN + FP) > 0 else float('nan')

print('Binary scalar metrics (excluding unknown search results):')
print(f'  N (binary subset) : {total}')
print(f'  TP (agree, adopt) : {TP}')
print(f'  TN (agree, no 311): {TN}')
print(f'  FP (search found, O\'Byrne no): {FP}')
print(f'  FN (search missed, O\'Byrne yes): {FN}')
print()
print(f'  agreement_rate : {agreement_rate:.3f}')
print(f'  conflict_rate  : {conflict_rate:.3f}')
print(f'  fn_rate        : {fn_rate:.3f}  (search missed adoptions O\'Byrne confirms)')
print(f'  fp_rate        : {fp_rate:.3f}  (search found adoption O\'Byrne says no)')

Binary scalar metrics (excluding unknown search results):
  N (binary subset) : 137
  TP (agree, adopt) : 42
  TN (agree, no 311): 39
  FP (search found, O'Byrne no): 56
  FN (search missed, O'Byrne yes): 0

  agreement_rate : 0.591
  conflict_rate  : 0.409
  fn_rate        : 0.000  (search missed adoptions O'Byrne confirms)
  fp_rate        : 0.589  (search found adoption O'Byrne says no)


In [6]:
# Conflict listing: only TRUE conflicts
# FP = year found <= 2012 but O'Byrne says not adopted (post-2012 adopters are NOT conflicts)
# FN = search found no_311 but O'Byrne says adopted
true_conflicts = binary[
    ((binary['obyrne_label'] == 'not_adopted_by2012') & (binary['search_class'] == 'year_found') & (binary['year_found'] <= 2012)) |
    ((binary['obyrne_label'] == 'adopted_by2012')     & (binary['search_class'] == 'no_311'))
].copy()

true_conflicts['conflict_type'] = true_conflicts.apply(
    lambda r: 'FP' if (r['obyrne_label'] == 'not_adopted_by2012' and r['search_class'] == 'year_found')
              else 'FN',
    axis=1
)

n_fp = len(true_conflicts[true_conflicts['conflict_type'] == 'FP'])
n_fn = len(true_conflicts[true_conflicts['conflict_type'] == 'FN'])
print(f'True conflict cities ({len(true_conflicts)} total — FP={n_fp}, FN={n_fn})')
print(f'(43 additional cities adopted after 2012; O\'Byrne "not adopted" is expected for them — not listed here)')
print()
cols_show = ['place_name', 'state_abbr', 'pop_2020', 'adoption_year_raw', 'obyrne_label', 'conflict_type', 'source_type', 'evidence_note']
for _, row in true_conflicts[cols_show].iterrows():
    print(f"  [{row['conflict_type']}] {row['place_name']}, {row['state_abbr']} | pop={row['pop_2020']:,}")
    print(f"        search={row['adoption_year_raw']} ({row['source_type']}) | O'Byrne={row['obyrne_label']}")
    print(f"        evidence: {row['evidence_note']}")
    print()

True conflict cities (13 total — FP=13, FN=0)
(43 additional cities adopted after 2012; O'Byrne "not adopted" is expected for them — not listed here)

  [FP] El Paso city, TX | pop=679,255
        search=2011 (news) | O'Byrne=not_adopted_by2012
        evidence: EP311 launched in 2011 as El Paso's non-emergency citizen service center.

  [FP] Boston city, MA | pop=675,466
        search=2009 (official) | O'Byrne=not_adopted_by2012
        evidence: Boston launched its 311 system (BOS:311) in 2009.

  [FP] Las Vegas city, NV | pop=646,794
        search=2000 (news) | O'Byrne=not_adopted_by2012
        evidence: Las Vegas Sun reported the city's 311 system went active on October 30, 2000.

  [FP] Milwaukee city, WI | pop=577,207
        search=2012 (news) | O'Byrne=not_adopted_by2012
        evidence: Milwaukee established its Unified Call Center / 311 system in 2012.

  [FP] Raleigh city, NC | pop=465,354
        search=2011 (news) | O'Byrne=not_adopted_by2012
        evidence: Raleigh 

In [7]:
# Assign audit_note for true conflict cities only
conflict_geoids = set(true_conflicts['GEOID'])

def audit_note(row):
    if row['GEOID'] not in conflict_geoids:
        return ''
    if row['obyrne_label'] == 'not_adopted_by2012' and row['search_class'] == 'year_found':
        yr = row['adoption_year_raw']
        return f"FP-conflict: search found year {yr} but O'Byrne (2012 survey) says not adopted. Verify source."
    if row['obyrne_label'] == 'adopted_by2012' and row['search_class'] == 'no_311':
        return "FN-conflict: search found no 311 system but O'Byrne (2012 survey) says adopted. City may have discontinued service or search missed it."
    return ''

merged['search_class'] = merged['adoption_year_raw'].apply(search_class)
merged['audit_note'] = merged.apply(audit_note, axis=1)

print(f'audit_note assigned to {(merged["audit_note"] != "").sum()} cities (the 13 true conflicts).')

audit_note assigned to 13 cities (the 13 true conflicts).


In [8]:
# Source-type breakdown among cities with confirmed year
with_year = merged[merged['search_class'] == 'year_found']
print('Source-type breakdown (cities with confirmed adoption year):')
src_counts = with_year['source_type'].value_counts()
print(src_counts.to_string())
pct_official = src_counts.get('official', 0) / len(with_year) * 100
print(f'\n  Total with year: {len(with_year)}')
print(f'  Official sources: {src_counts.get("official", 0)} ({pct_official:.1f}%)')

Source-type breakdown (cities with confirmed adoption year):
source_type
news         68
official     44
secondary     9

  Total with year: 121
  Official sources: 44 (36.4%)


In [9]:
# Year plausibility check: search year <= 2012 but O'Byrne says not_adopted_by2012
implausible = merged[
    (merged['search_class'] == 'year_found') &
    (merged['year_found'] <= 2012) &
    (merged['obyrne_label'] == 'not_adopted_by2012')
]
print(f'Year plausibility check:')
print(f'  Cities where search year <= 2012 but O\'Byrne says not_adopted: {len(implausible)}')
if len(implausible):
    print(implausible[['place_name', 'state_abbr', 'year_found', 'source_type', 'evidence_note']].to_string(index=False))

Year plausibility check:
  Cities where search year <= 2012 but O'Byrne says not_adopted: 13
        place_name state_abbr  year_found source_type                                                                                                                                             evidence_note
      El Paso city         TX      2011.0        news                                                                                 EP311 launched in 2011 as El Paso's non-emergency citizen service center.
       Boston city         MA      2009.0    official                                                                                                         Boston launched its 311 system (BOS:311) in 2009.
    Las Vegas city         NV      2000.0        news                                                                             Las Vegas Sun reported the city's 311 system went active on October 30, 2000.
    Milwaukee city         WI      2012.0        news                      

In [10]:
# Save scalar metrics summary
metrics = pd.DataFrame([{
    'metric': 'n_in_matrix',        'value': total,          'description': 'Cities in binary confusion matrix'
}, {
    'metric': 'TP',                 'value': TP,             'description': 'Both sources: adopted'
}, {
    'metric': 'TN',                 'value': TN,             'description': 'Both sources: not adopted'
}, {
    'metric': 'FP',                 'value': FP,             'description': 'Search found year; O\'Byrne says not adopted'
}, {
    'metric': 'FN',                 'value': FN,             'description': 'Search found no_311; O\'Byrne says adopted'
}, {
    'metric': 'agreement_rate',     'value': round(agreement_rate, 4), 'description': '(TP+TN)/total'
}, {
    'metric': 'conflict_rate',      'value': round(conflict_rate, 4),  'description': '(FP+FN)/total'
}, {
    'metric': 'fn_rate',            'value': round(fn_rate, 4),        'description': 'FN/(TP+FN): search missed O\'Byrne-confirmed adoptions'
}, {
    'metric': 'fp_rate',            'value': round(fp_rate, 4),        'description': 'FP/(TN+FP): search found year O\'Byrne says did not exist'
}, {
    'metric': 'n_year_found',       'value': (merged['search_class'] == 'year_found').sum(), 'description': 'Cities with adoption year'
}, {
    'metric': 'n_no_311',           'value': (merged['search_class'] == 'no_311').sum(),     'description': 'Cities confirmed no 311'
}, {
    'metric': 'n_unknown',          'value': (merged['search_class'] == 'unknown').sum(),    'description': 'Cities with 311 or unknown'
}, {
    'metric': 'pct_official_source','value': round(pct_official / 100, 4),                  'description': 'Fraction of year-found cities backed by official source'
}])

metrics.to_csv(CLEAN + 'search_audit_metrics.csv', index=False)
print('Saved search_audit_metrics.csv')
print(metrics.to_string(index=False))

Saved search_audit_metrics.csv
             metric    value                                              description
        n_in_matrix 137.0000                        Cities in binary confusion matrix
                 TP  42.0000                                    Both sources: adopted
                 TN  39.0000                                Both sources: not adopted
                 FP  56.0000              Search found year; O'Byrne says not adopted
                 FN   0.0000                Search found no_311; O'Byrne says adopted
     agreement_rate   0.5912                                            (TP+TN)/total
      conflict_rate   0.4088                                            (FP+FN)/total
            fn_rate   0.0000    FN/(TP+FN): search missed O'Byrne-confirmed adoptions
            fp_rate   0.5895 FP/(TN+FP): search found year O'Byrne says did not exist
       n_year_found 121.0000                                Cities with adoption year
           n_no_311  55

In [11]:
# Corrected confusion matrix: restrict to pre-2012 adoptions only.
# Cities that adopted after 2012 appear as FPs in the raw matrix but are NOT real conflicts —
# O'Byrne "not_adopted" simply means they hadn't adopted by his 2012 cutoff.
# Corrected scope: (year_found <= 2012) OR (no_311).

binary_corrected = binary[
    (binary['search_class'] == 'no_311') |
    ((binary['search_class'] == 'year_found') & (binary['year_found'] <= 2012))
].copy()

TP_c = len(binary_corrected[(binary_corrected['obyrne_label'] == 'adopted_by2012')     & (binary_corrected['search_class'] == 'year_found')])
TN_c = len(binary_corrected[(binary_corrected['obyrne_label'] == 'not_adopted_by2012') & (binary_corrected['search_class'] == 'no_311')])
FP_c = len(binary_corrected[(binary_corrected['obyrne_label'] == 'not_adopted_by2012') & (binary_corrected['search_class'] == 'year_found')])
FN_c = len(binary_corrected[(binary_corrected['obyrne_label'] == 'adopted_by2012')     & (binary_corrected['search_class'] == 'no_311')])

total_c = TP_c + TN_c + FP_c + FN_c
agreement_rate_c = (TP_c + TN_c) / total_c if total_c > 0 else float('nan')
conflict_rate_c  = (FP_c + FN_c) / total_c if total_c > 0 else float('nan')
fn_rate_c        = FN_c / (TP_c + FN_c) if (TP_c + FN_c) > 0 else float('nan')
fp_rate_c        = FP_c / (TN_c + FP_c) if (TN_c + FP_c) > 0 else float('nan')

fp_post2012 = FP - FP_c   # computed, not hardcoded

print('Corrected binary metrics (year_found restricted to <= 2012):')
print(f'  N : {total_c}   TP : {TP_c}   TN : {TN_c}   FP : {FP_c}   FN : {FN_c}')
print(f'  agreement_rate_c : {agreement_rate_c:.3f}')
print(f'  conflict_rate_c  : {conflict_rate_c:.3f}')
print(f'  fn_rate_c        : {fn_rate_c:.3f}')
print(f'  fp_rate_c        : {fp_rate_c:.3f}')
print()
print(f'Raw FP={FP} = {fp_post2012} post-2012 adopters (not real conflicts) + {FP_c} true conflicts (year<=2012).')

# Append corrected metrics to the metrics CSV (c9 already saved the raw metrics above)
corrected_rows = pd.DataFrame([
    {'metric': 'TP_corrected',             'value': TP_c,                    'description': 'Corrected TP (year<=2012 or no_311 only)'},
    {'metric': 'TN_corrected',             'value': TN_c,                    'description': 'Corrected TN'},
    {'metric': 'FP_corrected',             'value': FP_c,                    'description': 'Corrected FP (year<=2012, OByrne not adopted)'},
    {'metric': 'FN_corrected',             'value': FN_c,                    'description': 'Corrected FN'},
    {'metric': 'agreement_rate_corrected', 'value': round(agreement_rate_c, 4), 'description': '(TP_c+TN_c)/total_c'},
    {'metric': 'conflict_rate_corrected',  'value': round(conflict_rate_c, 4),  'description': '(FP_c+FN_c)/total_c'},
    {'metric': 'fn_rate_corrected',        'value': round(fn_rate_c, 4),        'description': 'FN_c/(TP_c+FN_c)'},
    {'metric': 'fp_rate_corrected',        'value': round(fp_rate_c, 4),        'description': 'FP_c/(TN_c+FP_c)'},
    {'metric': 'fp_post2012',              'value': fp_post2012,             'description': 'FPs that are post-2012 adopters (not real conflicts)'},
    {'metric': 'fp_true',                  'value': FP_c,                    'description': 'True FPs: year<=2012 but OByrne says not adopted'},
])
metrics_updated = pd.concat([metrics, corrected_rows], ignore_index=True)
metrics_updated.to_csv(CLEAN + 'search_audit_metrics.csv', index=False)
print('Appended corrected metrics to search_audit_metrics.csv')

Corrected binary metrics (year_found restricted to <= 2012):
  N : 92   TP : 40   TN : 39   FP : 13   FN : 0
  agreement_rate_c : 0.859
  conflict_rate_c  : 0.141
  fn_rate_c        : 0.000
  fp_rate_c        : 0.250

Raw FP=56 = 43 post-2012 adopters (not real conflicts) + 13 true conflicts (year<=2012).
Appended corrected metrics to search_audit_metrics.csv


## 3B-3. Assign status_final and assemble city_311_final.csv

In [12]:
def status_final(row):
    sc = row['search_class']
    ob = row['obyrne_label']

    if sc == 'year_found':
        if ob == 'not_adopted_by2012' and row['year_found'] <= 2012:
            return 'conflict_fp'
        return 'adopted_year_verified'

    if sc == 'unknown':
        return 'adopted_year_unknown'

    if sc == 'no_311':
        if ob == 'adopted_by2012':
            return 'conflict_fn'
        if ob == 'not_adopted_by2012':
            return 'not_adopted'
        if ob == 'unclear':
            return 'not_adopted_unclear'
        return 'not_adopted'   # not_in_obyrne

    return 'unknown'

merged['status_final'] = merged.apply(status_final, axis=1)

print('status_final distribution:')
print(merged['status_final'].value_counts().to_string())

status_final distribution:
status_final
adopted_year_unknown     138
adopted_year_verified    108
not_adopted               54
conflict_fp               13
not_adopted_unclear        1


In [13]:
# Assemble final columns
final_cols = [
    'GEOID', 'place_name', 'state_abbr', 'pop_2020',
    'adoption_year_raw', 'year_found',
    'status_final',
    'source_url', 'source_type', 'evidence_note',
    'obyrne_label', 'adopt_by2012',
    'audit_note'
]

final = merged[final_cols].copy()
final = final.rename(columns={'adoption_year_raw': 'adoption_year'})

final.to_csv(CLEAN + 'city_311_final.csv', index=False)
print(f'Saved {len(final)} rows → {CLEAN}city_311_final.csv')
print()
print('Column list:')
for c in final.columns:
    nn = final[c].notna().sum()
    print(f'  {c:<25} {nn:>4} non-null')

Saved 314 rows → ../data_clean/city_311_final.csv

Column list:
  GEOID                      314 non-null
  place_name                 314 non-null
  state_abbr                 314 non-null
  pop_2020                   314 non-null
  adoption_year              314 non-null
  year_found                 121 non-null
  status_final               314 non-null
  source_url                 314 non-null
  source_type                314 non-null
  evidence_note              314 non-null
  obyrne_label               314 non-null
  adopt_by2012               217 non-null
  audit_note                 314 non-null


In [14]:
# Save per-city audit flags (for search_audit.csv — subset with O'Byrne overlap)
audit_cols = [
    'GEOID', 'place_name', 'state_abbr', 'pop_2020',
    'adoption_year', 'year_found', 'search_class',
    'obyrne_label', 'adopt_by2012',
    'status_final', 'audit_note',
    'source_type', 'evidence_note'
]
audit_out = final.copy()
audit_out['search_class'] = merged['search_class']

audit_out[['GEOID', 'place_name', 'state_abbr', 'pop_2020',
           'adoption_year', 'year_found', 'search_class',
           'obyrne_label', 'adopt_by2012',
           'status_final', 'audit_note',
           'source_type', 'evidence_note']].to_csv(CLEAN + 'search_audit.csv', index=False)

print(f'Saved {len(audit_out)} rows → {CLEAN}search_audit.csv')
print()

# Final summary
print('=== STEP 3 COMPLETE ===')
print(f'Total cities       : {len(final)}')
print(f'adopted_year_verified: {(final["status_final"]=="adopted_year_verified").sum()}')
print(f'adopted_year_unknown : {(final["status_final"]=="adopted_year_unknown").sum()}')
print(f'not_adopted          : {(final["status_final"]=="not_adopted").sum()}')
print(f'conflict_fp          : {(final["status_final"]=="conflict_fp").sum()}')
print(f'conflict_fn          : {(final["status_final"]=="conflict_fn").sum()}')
print(f'not_adopted_unclear  : {(final["status_final"]=="not_adopted_unclear").sum()}')

Saved 314 rows → ../data_clean/search_audit.csv

=== STEP 3 COMPLETE ===
Total cities       : 314
adopted_year_verified: 108
adopted_year_unknown : 138
not_adopted          : 54
conflict_fp          : 13
conflict_fn          : 0
not_adopted_unclear  : 1
